# 2. Data Preprocessing

Data preprocessing involves preparing the cleaned dataset for machine learning. This includes identifying input features and the target variable, handling categorical data, preparing numerical features, and splitting the dataset into training and testing sets.


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

## 2.1 Loading the Cleaned Dataset

The cleaned dataset generated during EDA is loaded for further preprocessing and machine learning.

In [4]:
df = pd.read_csv("../data/processed/used_cars_cleaned.csv")

print("Dataset loaded successfully!")
print("Dataset Shape:", df.shape)

df.head()

Dataset loaded successfully!
Dataset Shape: (3577, 8)


,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner
0,Maruti 800 AC,2007,60000,70000,Petrol,Individual,Manual,First Owner
1,Maruti Wagon R LXI Minor,2007,135000,50000,Petrol,Individual,Manual,First Owner
2,Hyundai Verna 1.6 SX,2012,600000,100000,Diesel,Individual,Manual,First Owner
3,Datsun RediGO T Option,2017,250000,46000,Petrol,Individual,Manual,First Owner
4,Honda Amaze VX i-DTEC,2014,450000,141000,Diesel,Individual,Manual,Second Owner


## 2.2 Identifying Input Features and Target Variable

The target variable is the selling price of the used car. All remaining relevant columns will be considered input features for predicting the selling price.

In [5]:
target = "selling_price"

X = df.drop(columns=[target])
y = df[target]

print("Input Features:")
print(X.columns.tolist())

print("\nTarget Variable:")
print(target)

print("\nInput Shape:", X.shape)
print("Target Shape:", y.shape)

Input Features:
['name', 'year', 'km_driven', 'fuel', 'seller_type', 'transmission', 'owner']

Target Variable:
selling_price

Input Shape: (3577, 7)
Target Shape: (3577,)


## 2.3 Identifying Numerical and Categorical Features

The input features are divided into numerical and categorical variables to apply suitable preprocessing techniques.

In [6]:
numerical_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

print("Numerical Features:")
print(numerical_features)

print("\nCategorical Features:")
print(categorical_features)

Numerical Features:
['year', 'km_driven']

Categorical Features:
['name', 'fuel', 'seller_type', 'transmission', 'owner']


## 2.4 Brand Extraction

The car name contains both brand and model information. To simplify categorical feature processing, the first word of the car name is extracted as a new feature called `brand`. The original `name` column is then removed.

In [7]:
# Extract brand from car name
df["brand"] = df["name"].str.split().str[0]

# Remove original name column
df.drop(columns=["name"], inplace=True)

# Display updated dataset
df.head()

,year,selling_price,km_driven,fuel,seller_type,transmission,owner,brand
0,2007,60000,70000,Petrol,Individual,Manual,First Owner,Maruti
1,2007,135000,50000,Petrol,Individual,Manual,First Owner,Maruti
2,2012,600000,100000,Diesel,Individual,Manual,First Owner,Hyundai
3,2017,250000,46000,Petrol,Individual,Manual,First Owner,Datsun
4,2014,450000,141000,Diesel,Individual,Manual,Second Owner,Honda


In [8]:
print("Updated Dataset Shape:", df.shape)

print("\nBrand Examples:")
print(df["brand"].value_counts().head(10))

print("\nRemaining Columns:")
print(df.columns.tolist())

Updated Dataset Shape: (3577, 8)

Brand Examples:
brand
Maruti        1072
Hyundai        637
Mahindra       328
Tata           308
Ford           220
Honda          216
Toyota         170
Chevrolet      151
Renault        110
Volkswagen      93
Name: count, dtype: int64

Remaining Columns:
['year', 'selling_price', 'km_driven', 'fuel', 'seller_type', 'transmission', 'owner', 'brand']


## 2.5 Train-Test Split

The dataset is divided into training and testing sets. The training set is used to train the machine learning models, while the testing set is used to evaluate their performance on unseen data.

An 80:20 split is used, with a fixed random state to ensure reproducibility.

In [9]:
from sklearn.model_selection import train_test_split

# Define input features and target variable
X = df.drop(columns=["selling_price"])
y = df["selling_price"]

# Split dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training Features Shape:", X_train.shape)
print("Testing Features Shape:", X_test.shape)

print("Training Target Shape:", y_train.shape)
print("Testing Target Shape:", y_test.shape)

Training Features Shape: (2861, 7)
Testing Features Shape: (716, 7)
Training Target Shape: (2861,)
Testing Target Shape: (716,)


## 2.6 Building the Preprocessing Pipeline

Numerical and categorical features require different preprocessing techniques.

- Numerical features are processed using StandardScaler.
- Categorical features are converted into numerical representations using OneHotEncoder.

A ColumnTransformer is used to combine these transformations into a single preprocessing pipeline.

In [10]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

# Identify numerical and categorical features
numerical_features = ["year", "km_driven"]

categorical_features = [
    "fuel",
    "seller_type",
    "transmission",
    "owner",
    "brand"
]

# Numerical preprocessing
numerical_transformer = StandardScaler()

# Categorical preprocessing
categorical_transformer = OneHotEncoder(
    handle_unknown="ignore"
)

# Combine preprocessing steps
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

print("Preprocessing pipeline created successfully!")

Preprocessing pipeline created successfully!
